# 🎨 Vancouver Public Art — Geospatial Data Visualization

**Author:** Boris  
**Date:** 2026  
**Tools:** Python, Folium, Pandas  
**Data Source:** [City of Vancouver Open Data Portal](https://opendata.vancouver.ca/explore/dataset/public-art/)

---

### Objective

Use **Folium** and **Pandas** to visualize the distribution of **747 public artworks** across Vancouver's 22 neighbourhoods.

### What We Will Build

1. A **Choropleth map** showing art density by neighbourhood
2. **Clustered markers** for exploring individual artworks
3. Data analysis of art types and historical trends

---
## Step 1: Import Libraries

We use **Pandas** for data processing and **Folium** for creating interactive maps.

In [8]:
import json
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from collections import Counter

print('Libraries imported successfully!')

Libraries imported successfully!


---
## Step 2: Load the Datasets

We use two GeoJSON files downloaded from the Vancouver Open Data Portal:

| File | Description |
|------|-------------|
| `public-art.geojson` | 747 artworks with title, artist, type, year, and coordinates |
| `local-area-boundary.geojson` | Boundary polygons for Vancouver's 22 neighbourhoods |

In [9]:
# load the public art dataset
with open('public-art.geojson', encoding='utf-8') as f:
    art_data = json.load(f)

# load the neighbourhood boundary dataset
with open('local-area-boundary.geojson', encoding='utf-8') as f:
    boundary_data = json.load(f)

print(f'Public art: {len(art_data["features"])} artworks')
print(f'Boundaries: {len(boundary_data["features"])} neighbourhoods')

Public art: 747 artworks
Boundaries: 22 neighbourhoods


---
## Step 3: Explore the Data

Let's look at what information each artwork record contains.

In [10]:
# examine the fields available in one artwork record
sample = art_data['features'][5]['properties']

for key, value in sample.items():
    print(f'{key:30s} → {value}')

registryid                     → 759
title_of_work                  → Should I Be Worried?
artistprojectstatement         → None
type                           → Site-integrated work
status                         → In place
sitename                       → Southeast False Creek seawall
siteaddress                    → None
primarymaterial                → Backlit LED acrylic text units with associated electronics
url                            → https://covapp.vancouver.ca/PublicArtRegistry/ArtworkDetail.aspx?ArtworkId=759
photourl                       → None
ownership                      → City of Vancouver
neighbourhood                  → Mount Pleasant
locationonsite                 → Southeast False Creek seawall, east of the Cambie Bridge
geo_local_area                 → Mount Pleasant
descriptionofwork              → This public artwork was produced by local artist Justin Langlois as part of the City’s first Artist-in-Residence program. Since mid-2016, Justin has been working 

Each artwork has a **title**, **type**, **year of installation**, **neighbourhood**, and **GPS coordinates** — everything we need to build our map.

---
## Step 4: Count Artworks by Neighbourhood

Before mapping, we need to:
1. Fix a few naming mismatches between the two datasets
2. Count how many artworks are in each neighbourhood

In [4]:
# some neighbourhood names in the art data don't exactly match the boundary data
# for example: 'RileyPark' should be 'Riley Park'
name_fixes = {
    'RileyPark': 'Riley Park',
    'DowntownEastside': 'Downtown',
    'Stanley Park': 'West End',
}

# get the official boundary names
boundary_names = set(f['properties']['name'] for f in boundary_data['features'])

# count artworks per neighbourhood
art_counts = Counter()

for feat in art_data['features']:
    name = feat['properties'].get('neighbourhood') or 'Unknown'
    name = name_fixes.get(name, name)           # apply fix if needed
    if name not in boundary_names:
        name = 'Unknown'
    feat['properties']['_neighbourhood'] = name  # store cleaned name
    art_counts[name] += 1

# convert to DataFrame (exclude unknown)
art_df = pd.DataFrame(
    [(k, v) for k, v in art_counts.items() if k != 'Unknown'],
    columns=['name', 'art_count']
).sort_values('art_count', ascending=False)

print(art_df.to_string(index=False))

                    name  art_count
                Downtown        269
                West End         69
          Mount Pleasant         57
                Fairview         49
              Strathcona         31
             Shaughnessy         26
               Kitsilano         24
Kensington-Cedar Cottage         23
      Grandview-Woodland         21
                 Marpole         19
              Riley Park         14
            South Cambie         11
     Renfrew-Collingwood          8
        Hastings-Sunrise          7
                  Sunset          6
               Killarney          5
                Oakridge          4
       Dunbar-Southlands          2
         West Point Grey          2
           Arbutus Ridge          2
     Victoria-Fraserview          1
              Kerrisdale          1


**Key finding:** Downtown has by far the most public art (269 pieces), followed by Mount Pleasant (57) and Fairview (52). The eastern and southern neighbourhoods have significantly fewer installations.

---
## Step 5: Art Type Breakdown

What kinds of public art does Vancouver have?

In [5]:
# count by art type
types = Counter(f['properties'].get('type', 'Unknown') for f in art_data['features'])

type_df = pd.DataFrame(
    types.most_common(10),
    columns=['Art Type', 'Count']
)

print('Top 10 Public Art Types in Vancouver:\n')
print(type_df.to_string(index=False))

Top 10 Public Art Types in Vancouver:

                 Art Type  Count
                Sculpture    186
  Two-dimensional artwork    173
     Site-integrated work    118
                    Mural     87
               Media work     45
     Memorial or monument     32
               Totem pole     24
                   Relief     20
                   Mosaic     17
Fountain or water feature     16


---
## Step 6: Create a Base Map of Vancouver

First, let's create a simple **Folium** map centered on Vancouver using the light `Cartodb positron` tile style.

In [6]:
# define Vancouver's coordinates
vancouver_lat = 49.2527
vancouver_lon = -123.1207

# create the base map
vancouver_map = folium.Map(
    location=[vancouver_lat, vancouver_lon],
    zoom_start=13,
    tiles='Cartodb positron'
)

# display the map
vancouver_map

---
## Step 7: Add a Choropleth Layer

A **Choropleth map** shades each neighbourhood based on its art count.
Darker colours = more public art.

We use `folium.Choropleth()` with:
- `geo_data` → the boundary GeoJSON
- `data` → our art count DataFrame
- `key_on` → the GeoJSON property to match (`name`)

In [7]:
# start with a fresh map
vancouver_map = folium.Map(
    location=[vancouver_lat, vancouver_lon],
    zoom_start=13,
    tiles='Cartodb positron'
)

# create the choropleth layer
folium.Choropleth(
    geo_data=boundary_data,
    data=art_df,
    columns=['name', 'art_count'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.8,
    legend_name='Public Art Count by Neighbourhood'
).add_to(vancouver_map)

# display the choropleth map
vancouver_map

**Observation:** The choropleth clearly shows Downtown as the art hub of Vancouver, with the colour intensity dropping sharply as we move south and east.

---
## Step 8: Add Artwork Markers

Now let's add **clustered markers** so we can explore individual artworks.
Each marker shows the artwork's title, year, type, and a link to the Vancouver Art Registry.

We use `MarkerCluster` so the map stays clean when zoomed out — markers group into numbered clusters.

In [8]:
# instantiate a marker cluster
art_cluster = MarkerCluster().add_to(vancouver_map)

# loop through each artwork and add a marker
for feat in art_data['features']:
    # get coordinates
    geo_pt = feat['properties'].get('geo_point_2d', {})
    if not geo_pt:
        continue
    lat = geo_pt.get('lat')
    lon = geo_pt.get('lon')
    if not lat or not lon:
        continue

    # get artwork info
    p = feat['properties']
    title = p.get('title_of_work', 'Untitled')
    year = p.get('yearofinstallation', 'N/A')
    art_type = p.get('type', 'N/A')
    address = p.get('siteaddress', '')
    url = p.get('url', '')

    # build popup text
    popup_html = f'<b>{title}</b><br>'
    popup_html += f'Year: {year}<br>'
    popup_html += f'Type: {art_type}<br>'
    if address:
        popup_html += f'Address: {address}<br>'
    if url:
        popup_html += f'<a href="{url}" target="_blank">View in Registry</a>'

    # add marker to cluster
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=title
    ).add_to(art_cluster)

print(f'Added markers for artworks to the map')

Added markers for artworks to the map


---
## Step 9: Display the Final Map

Our completed map has:
- **Choropleth layer** showing art density by neighbourhood
- **Clustered markers** for exploring 747 individual artworks
- **Interactive popups** with artwork details and registry links

In [9]:
# display the final map with both choropleth and markers
vancouver_map

---
## Conclusion

### Key Findings

1. **Downtown dominates** — Over one-third of Vancouver's public art is concentrated in the Downtown area
2. **Art follows density** — Neighbourhoods with higher population density tend to have more public art
3. **East-West divide** — Western and central neighbourhoods have significantly more art than eastern areas like Killarney, Sunset, and Victoria-Fraserview
4. **Long history** — Vancouver's public art spans over a century (1901–2026)

### Tools Used

| Tool | Purpose |
|------|--------|
| **Pandas** | Data cleaning and aggregation |
| **Folium** | Interactive map with Choropleth and MarkerCluster |
| **GeoJSON** | Geographic boundary data from Vancouver Open Data |

### Data Source

All data is freely available at [opendata.vancouver.ca](https://opendata.vancouver.ca)